# Canada Immigration Analysis - Data Cleaning & EDA
Three-layer analytical chain: Selection (who got invited) → Flow (who landed) → Outcome (income).
Data sources documented in `data/SOURCES.md`. All raw files are Tab-separated, UTF-8 encoded.

## 1. Layer 1 Selection:  EE invitation data (By Province/Category)
source: `ee_invitations_by_province_category.csv`
Cleaning goals: convert suppression symbol `--` in `TOTAL` to numeric; drop redundant French columns.
Note: excludes category-based selection (CBS) draws introduced in 2023.

In [8]:
import pandas as pd
df = pd.read_csv("../data/raw/ee_invitations_by_province_category.csv",sep="\t",encoding="utf-8")
df.head()
# df.info()

,EN_YEAR,FR_ANNEÉ,EN_PROVINCE_TERRITORY,FR_PROVINCE_TERRITOIRE,EN_INVITATION_CATEGORY,FR_CATEGORIE_D'INVITATION,TOTAL
0,2015,2015,Alberta,Alberta,Canadian Experience Class,Catégorie de l'expérience canadienne,4635
1,2015,2015,Alberta,Alberta,Federal Skilled Trades,Métiers spécialisés,1385
2,2015,2015,Alberta,Alberta,Federal Skilled Worker,Travailleurs qualifiés,1970
3,2015,2015,British Columbia,Colombie-Britannique,Canadian Experience Class,Catégorie de l'expérience canadienne,1295
4,2015,2015,British Columbia,Colombie-Britannique,Federal Skilled Trades,Métiers spécialisés,200


In [9]:
#
# df[~df['TOTAL'].str.isdigit()]['TOTAL'].unique()
df['TOTAL'] = pd.to_numeric(df['TOTAL'],errors='coerce')

In [10]:
# df = df.drop(columns=["FR_ANNEÉ", "FR_PROVINCE_TERRITOIRE", "FR_CATEGORIE_D'INVITATION"])
df.head()

,EN_YEAR,FR_ANNEÉ,EN_PROVINCE_TERRITORY,FR_PROVINCE_TERRITOIRE,EN_INVITATION_CATEGORY,FR_CATEGORIE_D'INVITATION,TOTAL
0,2015,2015,Alberta,Alberta,Canadian Experience Class,Catégorie de l'expérience canadienne,4635.0
1,2015,2015,Alberta,Alberta,Federal Skilled Trades,Métiers spécialisés,1385.0
2,2015,2015,Alberta,Alberta,Federal Skilled Worker,Travailleurs qualifiés,1970.0
3,2015,2015,British Columbia,Colombie-Britannique,Canadian Experience Class,Catégorie de l'expérience canadienne,1295.0
4,2015,2015,British Columbia,Colombie-Britannique,Federal Skilled Trades,Métiers spécialisés,200.0


In [11]:
print("province/territory: \n", df["EN_PROVINCE_TERRITORY"].unique())
print("invitation category: \n", df["EN_INVITATION_CATEGORY"].unique())
print("year: \n",df["EN_YEAR"].min(), df["EN_YEAR"].max())


province/territory: 
 <StringArray>
[                      'Alberta',              'British Columbia',
                      'Manitoba',                 'New Brunswick',
     'Newfoundland and Labrador',         'Northwest Territories',
                   'Nova Scotia',                       'Nunavut',
                       'Ontario',          'Prince Edward Island',
 'Province/Territory not stated',                  'Saskatchewan',
                         'Yukon']
Length: 13, dtype: str
invitation category: 
 <StringArray>
[ 'Canadian Experience Class',     'Federal Skilled Trades',
     'Federal Skilled Worker', 'Provincial Nominee Program',
                 'Not Stated']
Length: 5, dtype: str
year: 
 2015 2026


In [12]:
# df.info()
print(df[df["EN_PROVINCE_TERRITORY"] == "Province/Territory not stated"]["TOTAL"].sum())
print(df[df["EN_INVITATION_CATEGORY"] == "Not Stated"]["TOTAL"].sum())
print(df["TOTAL"].sum())


0.0
0.0
766380.0


In [13]:
df.groupby("EN_YEAR")["TOTAL"].sum()

EN_YEAR
2015     21535.0
2016     25725.0
2017     67745.0
2018     71725.0
2019     71630.0
2020     81150.0
2021    108100.0
2022     37320.0
2023     85745.0
2024     90005.0
2025     74610.0
2026     31090.0
Name: TOTAL, dtype: float64

## 2. Layer 1 slection: EE invitation data - by score range
source: `ee_ita_score_breakdown.csv`
Adds a CRS/ITA score-band dimension to ①, used to analyze selection thresholds and score distribution.
Cleaning goals: handle `--`; drop French columns. More suppression expected after score-band split.

## 3. Layer 2 — Flow: PR admissions (by province/immigration category)
Source: `pr_admissions_by_province_immcat.csv`
Actual permanent-resident landings, at month × province × category granularity.
Cleaning goals: handle `--`; drop French columns. Note: category names differ from ① (need mapping).

## 4. Layer 2 — Flow: PR admissions (by country of citizenship)

Source: `pr_admissions_by_citizenship.csv`
Source-country distribution, at month × province × citizenship granularity (~200 countries, largest file).
Cleaning goals: handle `--`; drop French columns. No category dimension — parallel view to ③.
